# Pipeline de Datos y Entrenamiento ML - Gaply Matchmaking
Este notebook implementa el flujo completo en 4 etapas separadas:
1. **Etapa 1:** Generación de sets de datos crudos homogéneos (`perfiles_crudos.csv` y `pares_interacciones_crudos.csv`).
2. **Etapa 2:** Procesamiento y normalización (`MinMaxScaler`) exportando `dataset_normalizado_balanceado.csv`.
3. **Etapa 3:** Entrenamiento supervisado desde el dataset procesado y persistencia del modelo (`.pkl`).
4. **Etapa 4:** Inferencia en vivo cargando el modelo entrenado desde disco sin reentrenar.

In [ ]:
# ==============================================================================
# CELDA 1: IMPORTACIONES Y CREACIÓN DE ESTRUCTURA DE CARPETAS DEL REPOSITORIO
# ==============================================================================
import os
import random
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

SEMILLA = x
print('Celda 1 lista · Estructura de carpetas creada (data/raw, data/processed, models)')

In [ ]:
# ==============================================================================
# CELDA 2 (ETAPA 1): GENERACIÓN Y GUARDADO DE SETS DE DATOS CRUDOS (RAW DATA)
# ==============================================================================
rng = random.Random(SEMILLA)
np.random.seed(SEMILLA)

UNIVERSIDADES = [
    'Universidad Jorge Tadeo Lozano',
    'Universidad Central',
    'Universidad de los Andes',
    'Otra',
]

CARRERAS = [
    'Ingeniería de Sistemas', 'Diseño Gráfico', 'Medicina', 'Derecho',
    'Psicología', 'Administración de Empresas', 'Ingeniería Industrial',
    'Historia del Arte', 'Comunicación Social', 'Economía', 'Química',
    'Arquitectura',
]

INTERESES = [
    'Deportes', 'Música', 'Videojuegos', 'Artes', 'Juegos de mesa',
    'Películas', 'Estudio', 'Eventos', 'Baile', 'Fotografía', 'Viajes',
    'Cocina', 'Programación', 'Moda', 'Ciencia',
]

NOMBRES = ['Santiago', 'Valentina', 'Camila', 'Andrés', 'Laura', 'Tomás',
           'Mariana', 'Nicolás', 'Daniela', 'Felipe', 'Paula', 'Julián']
APELLIDOS = ['Martínez', 'Rojas', 'Ortiz', 'Cárdenas', 'Restrepo', 'Herrera',
             'Gil', 'Pardo', 'Vargas', 'Gómez', 'Mejía', 'Castro']

N_PERFILES = x
MIN_INTERESES = x
MAX_INTERESES = x

# 1. Generación de Perfiles Homogéneos
perfiles = []
for i in range(N_PERFILES):
    uni = UNIVERSIDADES[i % len(UNIVERSIDADES)]  # 25% exacto por universidad
    carrera = CARRERAS[i % len(CARRERAS)]        # Distribución homogénea por carrera
    k_int = rng.randint(MIN_INTERESES, MAX_INTERESES)
    intereses_elegidos = rng.sample(INTERESES, k=k_int) # Muestreo uniforme sin sesgo

    perfiles.append({
        'user_id': f'u{i:05d}',
        'nombre': f'{rng.choice(NOMBRES)} {rng.choice(APELLIDOS)}',
        'university': uni,
        'career': carrera,
        'interests': '|'.join(intereses_elegidos),
        'num_interests': k_int
    })

df_perfiles = pd.DataFrame(perfiles).sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
df_perfiles.to_csv('data/raw/perfiles_crudos.csv', index=False)

# 2. Generación de Pares Crudos Balanceados (50% Match Ideal / 50% No Match Ideal) + Ruido Controlado
positivos_ideales = []
negativos_ideales = []
TARGET_POR_CLASE = 10000
RUIDO_PROBABILIDAD = 0.22  # 22% de ruido por factores humanos no observados

perfiles_list = df_perfiles.to_dict('records')

while len(positivos_ideales) < TARGET_POR_CLASE or len(negativos_ideales) < TARGET_POR_CLASE:
    u_a, u_b = rng.sample(perfiles_list, 2)
    set_a = set(u_a['interests'].split('|'))
    set_b = set(u_b['interests'].split('|'))

    intereses_comunes = len(set_a.intersection(set_b))
    union_intereses = len(set_a.union(set_b))
    jaccard = round(intereses_comunes / union_intereses, 4)
    misma_uni = 1 if u_a['university'] == u_b['university'] else 0
    misma_carrera = 1 if u_a['career'] == u_b['career'] else 0

    score_base = intereses_comunes * 1.0 + misma_uni * 1.5 + misma_carrera * 0.5
    match_ideal = 1 if score_base >= 2.5 else 0

    if match_ideal == 1 and len(positivos_ideales) < TARGET_POR_CLASE:
        es_ruido = 1 if rng.random() < RUIDO_PROBABILIDAD else 0
        is_match = 0 if es_ruido else 1
        positivos_ideales.append({
            'user_a_id': u_a['user_id'], 'user_b_id': u_b['user_id'],
            'misma_universidad': misma_uni, 'misma_carrera': misma_carrera,
            'intereses_en_comun': intereses_comunes, 'similitud_jaccard': jaccard,
            'is_match': is_match
        })
    elif match_ideal == 0 and len(negativos_ideales) < TARGET_POR_CLASE:
        es_ruido = 1 if rng.random() < RUIDO_PROBABILIDAD else 0
        is_match = 1 if es_ruido else 0
        negativos_ideales.append({
            'user_a_id': u_a['user_id'], 'user_b_id': u_b['user_id'],
            'misma_universidad': misma_uni, 'misma_carrera': misma_carrera,
            'intereses_en_comun': intereses_comunes, 'similitud_jaccard': jaccard,
            'is_match': is_match
        })

df_pares_crudos = pd.DataFrame(positivos_ideales + negativos_ideales)
df_pares_crudos = df_pares_crudos.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
df_pares_crudos.to_csv('data/raw/pares_interacciones_crudos.csv', index=False)

print(f'Archivo guardado: data/raw/perfiles_crudos.csv ({len(df_perfiles)} filas)')
print(f'Archivo guardado: data/raw/pares_interacciones_crudos.csv ({len(df_pares_crudos)} filas)')
print('\nDistribución por universidad (Homogeneidad verificada):')
print(df_perfiles['university'].value_counts().to_string())
print('\nBalance de clases en pares crudos:')
print(df_pares_crudos['is_match'].value_counts(normalize=True).round(4).to_string())

In [ ]:
# ==============================================================================
# CELDA 3 (ETAPA 2): LECTURA DE DATOS CRUDOS, NORMALIZACIÓN Y EXPORTACIÓN
# ==============================================================================
df_raw_pairs = pd.read_csv('data/raw/pares_interacciones_crudos.csv')

FEATURES = ['misma_universidad', 'misma_carrera', 'intereses_en_comun', 'similitud_jaccard']
TARGET = 'is_match'

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_raw_pairs[FEATURES])

df_normalizado = pd.DataFrame(np.round(X_scaled, 4), columns=FEATURES)
df_normalizado[TARGET] = df_raw_pairs[TARGET]

df_normalizado.to_csv('data/processed/dataset_normalizado_balanceado.csv', index=False)
joblib.dump(scaler, 'models/scaler_normalizador.pkl')

print('Archivo guardado: data/processed/dataset_normalizado_balanceado.csv')
print('Escalador guardado: models/scaler_normalizador.pkl')
print('\nVista previa de datos normalizados (rango 0.0 a 1.0):')
print(df_normalizado.head())

In [ ]:
# ==============================================================================
# CELDA 4 (ETAPA 3): ENTRENAMIENTO DESDE EL ARCHIVO PROCESADO Y GUARDADO (.PKL)
# ==============================================================================
df_train_data = pd.read_csv('data/processed/dataset_normalizado_balanceado.csv')

X = df_train_data[FEATURES]
y = df_train_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMILLA, stratify=y
)

modelos = {
    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=SEMILLA),
    'gradient_boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=SEMILLA),
    'logistic_regression': LogisticRegression(class_weight='balanced', random_state=SEMILLA)
}

resultados = {}
mejor_nombre = None
mejor_acc = -1.0
mejor_modelo_obj = None

for nombre, mod in modelos.items():
    mod.fit(X_train, y_train)
    preds = mod.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    resultados[nombre] = (acc, prec, rec, f1)
    if acc > mejor_acc:
        mejor_acc = acc
        mejor_nombre = nombre
        mejor_modelo_obj = mod

joblib.dump(mejor_modelo_obj, 'models/modelo_matchmaking_gaply.pkl')

print('=======================================================')
print('RESULTADOS DEL ENTRENAMIENTO')
print('=======================================================')
print(f'Mejor modelo: {mejor_nombre}')
print(f'Accuracy: {mejor_acc:.4f} ({mejor_acc*100:.2f}%)')
print(f'Datos de entrenamiento: {len(X_train)}')
print(f'Datos de prueba: {len(X_test)}')
print('\nMetricas por modelo:')
for nombre, (acc, prec, rec, f1) in resultados.items():
    print(f'\n{nombre}:')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1:        {f1:.4f}')
print('\nModelo entrenado guardado exitosamente en: models/modelo_matchmaking_gaply.pkl')

In [ ]:
# ==============================================================================
# CELDA 5 (ETAPA 4): PRUEBA DE INFERENCIA EN VIVO CARGANDO EL MODELO GUARDADO
# ==============================================================================
modelo_cargado = joblib.load('models/modelo_matchmaking_gaply.pkl')
scaler_cargado = joblib.load('models/scaler_normalizador.pkl')

# Simulamos dos estudiantes nuevos sin reentrenar el modelo
nuevo_par_crudo = pd.DataFrame([{
    'misma_universidad': 1,
    'misma_carrera': 0,
    'intereses_en_comun': 3,
    'similitud_jaccard': 0.50
}])

nuevo_par_norm = pd.DataFrame(
    scaler_cargado.transform(nuevo_par_crudo[FEATURES]),
    columns=FEATURES
)

prediccion = modelo_cargado.predict(nuevo_par_norm)[0]
probabilidad = modelo_cargado.predict_proba(nuevo_par_norm)[0][1]

print(f'Predicción del modelo guardado: {"MATCH (1)" if prediccion == 1 else "NO MATCH (0)"}')
print(f'Probabilidad de compatibilidad: {probabilidad*100:.2f}%')